# Module 2: Initial Price Push (Daily 6 AM Reset)

## Purpose
This module runs once daily at 8 AM Cairo time to:
1. **Load and prepare data** from Snowflake (MATERIALIZED_VIEWS.Pricing_data_extraction)
2. Reset prices for all SKUs based on ABC classification
3. Set initial cart rules based on normal_refill and stddev
4. Apply status-based adjustments (combined_status + yesterday_status)
5. Push cart rules and prices via API

## Data Flow
```
data_extraction.ipynb → Snowflake (Pricing_data_extraction) → Module 2 (this module)
                                                        ├── Data Preparation
                                                        ├── Price Logic
                                                        ├── Cart Rule Logic
                                                        └── Push to API
```

## Price Setting Logic
- **Zero demand SKUs**: Market minimum price + SKU discount
- **With market data**: A=25th percentile, B=50th, C=75th
- **Without market data**: A=50% margin range, B=75%, C=90%
- **No data SKUs**: Average margin of their category

## Status Adjustment Logic
- Both below On Track: -1 step from current price
- Both above On Track: +1 step from current price
- Combined lower, Yesterday higher: No action (oscillation prevention)
- Combined higher, Yesterday lower: No action (trend observation)
- On Track: No action
- Above On Track + Yesterday On Track: No action


In [1]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import sys
sys.path.append('..')

%run queries_module.ipynb
# Cairo timezone
CAIRO_TZ = pytz.timezone('Africa/Cairo')
CAIRO_NOW = datetime.now(CAIRO_TZ)
TODAY = CAIRO_NOW.date()
CURRENT_HOUR = CAIRO_NOW.hour

# Configuration constants
# Cart rule constraints
MIN_CART_RULE = 10  # Minimum for Module 2 (fallback when no percentile data)
MAX_CART_RULE = 500
LOW_STOCK_DOH_THRESHOLD = 1  # SKUs with DOH <= this are protected from price reduction
STATUS_BELOW_ON_TRACK = ['No Data', 'Critical', 'Struggling', 'Underperforming']
STATUS_ABOVE_ON_TRACK = ['Over Achiever', 'Star Performer']
STATUS_ON_TRACK = ['On Track']

# Input/Output configuration
# Data is now loaded from Snowflake instead of Excel
INPUT_TABLE = 'MATERIALIZED_VIEWS.Pricing_data_extraction'
OUTPUT_FILE = f'module_2_output_{CAIRO_NOW.strftime("%Y%m%d_%H%M")}.xlsx'

print(f"Module 2: Initial Price Push")
print(f"Run Time (Cairo): {CAIRO_NOW.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Input: {INPUT_TABLE} (today's data)")
print(f"Output: {OUTPUT_FILE}")


/home/ec2-user/.Renviron


/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()            - Fixed prices from Google Sheet

Commercial Constraints:
  • get_commercial_min_prices()   - Fresh min price constrai

In [2]:
# =============================================================================
# LOAD DATA FROM SNOWFLAKE (Instead of Excel file)
# =============================================================================
print("Loading data from Snowflake...")

# Query to get today's data from Pricing_data_extraction
LOAD_QUERY = f"""
SELECT * FROM {INPUT_TABLE}
WHERE created_at = '{datetime.now(CAIRO_TZ).date()}'
"""

df_raw = query_snowflake(LOAD_QUERY)
print(f"Loaded {len(df_raw)} records from Snowflake")

# -------------------------------------------------------------------------
# DATA PREPARATION: Transform raw data to structured format
# This replicates the data structuring from pricing_action_engine.ipynb
# -------------------------------------------------------------------------
print("\nPreparing data (structuring columns, handling nulls)...")

# Create a clean DataFrame with all required columns and proper defaults
df = pd.DataFrame()

# Identifiers
df['warehouse_id'] = df_raw['warehouse_id']
df['product_id'] = df_raw['product_id']
df['sku'] = df_raw['sku']
df['cohort_id'] = df_raw['cohort_id'] if 'cohort_id' in df_raw.columns else None

# Product info
df['abc_class'] = df_raw['abc_class'].fillna('C')
df['brand'] = df_raw['brand'] if 'brand' in df_raw.columns else None
df['cat'] = df_raw['cat'] if 'cat' in df_raw.columns else None
df['sensitivity'] = df_raw['sensitivity'] if 'sensitivity' in df_raw.columns else None

# Current state - with null handling
df['current_price'] = pd.to_numeric(df_raw['current_price'], errors='coerce').fillna(0)
df['current_cart_rule'] = pd.to_numeric(df_raw['current_cart_rule'], errors='coerce').fillna(999)
df['normal_refill'] = pd.to_numeric(df_raw.get('normal_refill', 0), errors='coerce').fillna(0)
df['refill_stddev'] = pd.to_numeric(df_raw.get('refill_stddev', 0), errors='coerce').fillna(0)
df['wac_p'] = pd.to_numeric(df_raw['wac_p'], errors='coerce').fillna(0)
df['commercial_min_price'] = pd.to_numeric(df_raw.get('commercial_min_price', 0), errors='coerce').fillna(0)

# Performance status
df['combined_status'] = df_raw['combined_status'].fillna('No Data')
df['yesterday_status'] = df_raw['yesterday_status'].fillna('No Data')
df['oos_yesterday'] = df_raw['oos_yesterday'].fillna(0).astype(int)

# Stock and demand
df['stocks'] = pd.to_numeric(df_raw['stocks'], errors='coerce').fillna(0)
df['zero_demand'] = df_raw['zero_demand'].fillna(0).astype(int)
df['doh'] = pd.to_numeric(df_raw.get('doh', 999), errors='coerce').fillna(999)  # Days on Hand for low stock protection

# Margin data (for price tier calculations)
df['target_margin'] = pd.to_numeric(df_raw.get('target_margin', 0), errors='coerce').fillna(0)
#df['target_margin_std'] = pd.to_numeric(df_raw.get('target_margin_std', 0), errors='coerce').fillna(0)

# Market margins (for price tiers)
market_margin_cols = ['below_market', 'market_min', 'market_25', 'market_50', 
                      'market_75', 'market_max', 'above_market']
for col in market_margin_cols:
    if col in df_raw.columns:
        df[col] = pd.to_numeric(df_raw[col], errors='coerce')
    else:
        df[col] = np.nan

# Internal margin tiers
margin_tier_cols = ['margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
                    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2']
for col in margin_tier_cols:
    if col in df_raw.columns:
        df[col] = pd.to_numeric(df_raw[col], errors='coerce')
    else:
        df[col] = np.nan

# All-time high margin (price ceiling for increases)
df['all_time_high_margin'] = pd.to_numeric(df_raw.get('all_time_high_margin', np.nan), errors='coerce')

# P80/P70 for cart rules fallback
df['p80_daily_240d'] = pd.to_numeric(df_raw.get('p80_daily_240d', 0), errors='coerce').fillna(0)
df['p70_daily_retailers_240d'] = pd.to_numeric(df_raw.get('p70_daily_retailers_240d', 1), errors='coerce').fillna(1)

print(f"✅ Data prepared: {len(df)} records")
print(f"\nABC Class Distribution:")
print(df['abc_class'].value_counts().to_string())
print(f"\nCombined Status Distribution:")
print(df['combined_status'].value_counts().to_string())

# =============================================================================
# LOAD PERCENTILE DATA FOR CART RULES
# =============================================================================
df_percentiles = get_percentile_data()

# =============================================================================
# LOAD MARKET DATA V2 + SIGNALS
# =============================================================================
%run market_data_module_2.ipynb

# Get V2 price tiers and merge into df
df_market_v2 = get_market_data_v2()
df_market_cohorts = expand_to_cohorts(df_market_v2)
df = df.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'market_data_source']],
    on=['product_id', 'cohort_id'], how='left'
)
df['price_tiers'] = df['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])
print(f'  V2 price tiers merged: {(df["price_tiers"].apply(len) > 0).sum()} SKUs with tiers')

# Get margin tiers and build margin_tier_prices list per row
df_margin_tiers_data = get_margin_tiers()
margin_tier_cols = ['margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
                    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2']
df = df.merge(
    df_margin_tiers_data[['product_id', 'warehouse_id'] + margin_tier_cols],
    on=['product_id', 'warehouse_id'], how='left'
)

def build_margin_tier_prices(row):
    wac = row.get('wac_p', 0)
    if wac <= 0:
        return []
    prices = []
    for col in margin_tier_cols:
        m = row.get(col)
        if pd.notna(m) and 0 < m < 1:
            prices.append(round(wac / (1 - m) * 4) / 4)
    return sorted(set(prices))

df['margin_tier_prices'] = df.apply(build_margin_tier_prices, axis=1)

# Build effective_tiers: price_tiers > margin_tier_prices > empty
def get_effective_tiers(row):
    if row['price_tiers'] and len(row['price_tiers']) > 0:
        return row['price_tiers']
    if row['margin_tier_prices'] and len(row['margin_tier_prices']) > 0:
        return row['margin_tier_prices']
    return []

df['effective_tiers'] = df.apply(get_effective_tiers, axis=1)
print(f'  Effective tiers: {(df["effective_tiers"].apply(len) > 0).sum()} SKUs with any tiers')

# Market signals
df_market_signals = get_market_signals()

# Commercial price-up forecasts (fallback signal)
df_price_ups = get_commercial_price_ups()
if len(df_price_ups) > 0:
    df = df.merge(
        df_price_ups[['product_id', 'diff']].rename(columns={'diff': 'commercial_price_up_pct'}),
        on='product_id', how='left'
    )
    print(f'  Commercial price-ups merged: {df["commercial_price_up_pct"].notna().sum()} SKUs')
else:
    df['commercial_price_up_pct'] = 0
    print('  No commercial price-ups found')

if len(df_market_signals) > 0:
    signal_cols = ['product_id', 'warehouse_id', 'trend_signal', 'momentum', 
                   'pct_change_30d', 'sma_30d', 'data_points_30d', 'volatility_pct']
    signal_cols = [c for c in signal_cols if c in df_market_signals.columns]
    df = df.merge(df_market_signals[signal_cols], on=['product_id', 'warehouse_id'], how='left')
    df['trend_signal'] = df['trend_signal'].fillna('NEUTRAL')
    df['data_points_30d'] = df.get('data_points_30d', pd.Series(dtype=float)).fillna(0)
    df['volatility_pct'] = df.get('volatility_pct', pd.Series(dtype=float)).fillna(0)
    print(f"\n  Market signals merged: {df['trend_signal'].value_counts().to_string()}")
else:
    df['trend_signal'] = 'NEUTRAL'
    df['data_points_30d'] = 0
    df['volatility_pct'] = 0
    print("  No market signals available - all set to NEUTRAL")


Loading data from Snowflake...


Loaded 29967 records from Snowflake

Preparing data (structuring columns, handling nulls)...
✅ Data prepared: 29967 records

ABC Class Distribution:
abc_class
C    24405
B     4814
A      748

Combined Status Distribution:
combined_status
No Data            10923
Critical            4421
Struggling          4231
Underperforming     3178
On Track            2972
Over Achiever       2233
Star Performer      2009
Fetching percentile data for cart rules...


  Loaded 18575 percentile records
   Percentiles available for 3464 unique products


/home/ec2-user/.Renviron


/home/ec2-user/service_account_key.json


...

  Loaded 19168 product-region margin boundary records
    After region fallback: 6322 still bad
Fetching global-level margin boundaries...


  Loaded 4293 product-level margin boundary records
    After global fallback: 2876 still bad
    Fallback summary: 2033 region, 6322 global

Step 3: Calculating margin tiers...

MARGIN TIERS COMPLETE
Total records: 43851


  Effective tiers: 24991 SKUs with any tiers

FETCHING MARKET SIGNALS (Trend Analysis)
Timestamp: 2026-04-21 08:18:30 Cairo time
Analyzing 60-day price history from Pricing_data_extraction...



  Total SKUs analyzed: 27004

  --- Trend Distribution ---
    STRONG UPTREND           : 8454 SKUs
    WEAK UPTREND             : 6727 SKUs
    UPTREND                  : 4230 SKUs
    WEAK DOWNTREND           : 2694 SKUs
    DOWNTREND                : 2302 SKUs
    STRONG DOWNTREND         : 2107 SKUs
    SIDEWAYS                 :  490 SKUs

  --- Market Overview ---
    Avg 30d price change:   +5.21%
    SKUs up >5% (30d):      10652
    SKUs down >5% (30d):    1038

MARKET SIGNALS COMPLETE
Fetching commercial price-up forecasts...


  Loaded 171 price-up forecasts
  Commercial price-ups merged: 1675 SKUs

  Market signals merged: trend_signal
STRONG UPTREND      8288
WEAK UPTREND        6581
UPTREND             4051
NEUTRAL             3910
WEAK DOWNTREND      2599
DOWNTREND           2154
STRONG DOWNTREND    2016
SIDEWAYS             368


In [3]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

# Minimum price change constant (ensure it's defined in this cell for function access)
MIN_PRICE_CHANGE_EGP = 0.25  # Minimum 0.25 EGP for any price increase or decrease

def is_below_on_track(status):
    """Check if status is below On Track."""
    return str(status).strip() in STATUS_BELOW_ON_TRACK

def is_above_on_track(status):
    """Check if status is above On Track."""
    return str(status).strip() in STATUS_ABOVE_ON_TRACK

def is_on_track(status):
    """Check if status is On Track."""
    return str(status).strip() in STATUS_ON_TRACK

def calculate_margin(price, wac):
    """Calculate margin from price and WAC."""
    if pd.isna(price) or pd.isna(wac) or price == 0:
        return None
    return (price - wac) / price

def find_next_price_above(current_price, row):
    """Find the first tier ABOVE current_price by at least MIN_PRICE_CHANGE_EGP.
    Uses effective_tiers (price_tiers > margin_tier_prices)."""
    if pd.isna(current_price) or current_price <= 0:
        return current_price
    
    tiers = row.get('effective_tiers', [])
    if not tiers:
        return current_price
    
    for tier in tiers:
        if tier > current_price + MIN_PRICE_CHANGE_EGP:
            return round(tier, 2)
    
    return current_price

def get_above_market_price(current_price, row):
    """
    Fallback price when all tier ladders are exhausted.
    Uses 3-tier chain: avg margin step → 20% target margin → +1% bump.
    """
    wac = row.get('wac_p', 0)
    if wac <= 0 or current_price <= 0:
        return current_price
    
    current_margin = (current_price - wac) / current_price
    
    # Fallback 1: average margin step from effective tiers
    all_prices = sorted(set(p for p in row.get('effective_tiers', []) if p > wac))
    if len(all_prices) >= 2:
        margins = [(p - wac) / p for p in all_prices]
        steps = [margins[i+1] - margins[i] for i in range(len(margins)-1)]
        new_margin = current_margin + np.mean(steps)
        if new_margin < 0.99:
            new_price = round(wac / (1 - new_margin) * 4) / 4
            if new_price > current_price:
                return new_price
    
    # Fallback 2: 20% of target_margin as step
    target_margin = row.get('target_margin', 0)
    if pd.notna(target_margin) and target_margin > 0:
        new_margin = current_margin + target_margin * 0.20
        if new_margin < 0.99:
            new_price = round(wac / (1 - new_margin) * 4) / 4
            if new_price > current_price:
                return new_price
    
    # Fallback 3: +1% price bump
    return round(current_price * 1.01 * 4) / 4

def find_next_price_below(current_price, row):
    """Find the first tier BELOW current_price by at least MIN_PRICE_CHANGE_EGP.
    Uses effective_tiers (price_tiers > margin_tier_prices)."""
    if pd.isna(current_price) or current_price <= 0:
        return current_price
    
    tiers = row.get('effective_tiers', [])
    if not tiers:
        return current_price
    
    for tier in reversed(tiers):
        if tier < current_price - MIN_PRICE_CHANGE_EGP:
            return round(tier, 2)
    
    return current_price

print("Helper functions loaded.")


Helper functions loaded.


In [4]:
# =============================================================================
# STATUS ADJUSTMENT & PRICE FUNCTIONS
# =============================================================================

def get_price_action(combined_status, yesterday_status):
    """
    Determine price action based on status combination.
    
    Returns:
        str: 'increase', 'decrease', or 'hold'
        str: Reason for action
    """
    combined_below = is_below_on_track(combined_status)
    combined_above = is_above_on_track(combined_status)
    combined_on = is_on_track(combined_status)
    yesterday_below = is_below_on_track(yesterday_status)
    yesterday_above = is_above_on_track(yesterday_status)
    yesterday_on = is_on_track(yesterday_status)
    
    # On Track = no action
    if combined_on and yesterday_on:
        return 'hold', "On Track - no price change"
    if combined_on and yesterday_above:
        return 'increase', f"yesterday above - combined on  ({combined_status}, {yesterday_status}) - increase"
    
    # Both ABOVE On Track: INCREASE price (only if both are above, not on track)
    if combined_above and yesterday_above:
        return 'increase', f"Both above ({combined_status}, {yesterday_status}) - increase"
    
    # Combined above, Yesterday on track: HOLD (changed from increase)
    if combined_above and yesterday_on:
        return 'hold', f"Above + On Track ({combined_status}, {yesterday_status}) - hold"
    
    # Both below On Track: DECREASE price (go to first tier below current)
    if combined_below and yesterday_below:
        return 'decrease', f"Both below/on ({combined_status}, {yesterday_status}) - decrease"
    
    # Combined below, Yesterday above: No action (oscillation prevention)
    if combined_below and yesterday_above:
        return 'hold', f"Oscillation prevention ({combined_status} vs {yesterday_status}) - hold"
    
    # Combined above, Yesterday below: HOLD (observe trend before reacting)
    if (combined_above and yesterday_below) or (combined_below and yesterday_on):
        return 'hold', f"Trend observation ({combined_status} vs {yesterday_status}) - hold"
    
    return 'hold', "Default - no price change"

def get_margin_increase_pct(combined_status, yesterday_status):
    """
    Determine margin increase as a % of target_margin when no tier above is found.
    Used as a fallback for price increases.
    """
    c = str(combined_status).strip()
    y = str(yesterday_status).strip()
    
    if c == 'Star Performer' and y == 'Star Performer':
        return 0.15
    if is_above_on_track(c) and is_above_on_track(y):
        return 0.10
    if is_on_track(c) and is_above_on_track(y):
        return 0.05
    return 0


def apply_price_action(current_price, action, row, margin_increase_pct=None):
    """
    Apply price action: find next tier above/below current price.
    Falls back to margin-based increase (% of target_margin) when no tier above is found.
    
    Args:
        current_price: Current SKU price
        action: 'increase', 'decrease', or 'hold'
        row: DataFrame row with tier data
        margin_increase_pct: Optional fallback increase as fraction of target_margin
    
    Returns:
        float: New price
        str: Source of new price (market/margin/unchanged)
    """
    if action == 'hold' or pd.isna(current_price):
        return current_price, 'unchanged'
    
    if action == 'increase':
        new_price = find_next_price_above(current_price, row)
        if new_price > current_price:
            source = 'tiers'
            return new_price, source
        
        # Fallback: margin-based increase when no tier above found
        if margin_increase_pct and margin_increase_pct > 0:
            target_margin = row.get('target_margin', 0)
            wac = row.get('wac_p', 0)
            if pd.notna(target_margin) and target_margin > 0 and wac > 0 and current_price > 0:
                current_margin = (current_price - wac) / current_price
                margin_step = target_margin * margin_increase_pct
                new_margin = min(current_margin + margin_step, 0.99)
                if new_margin > current_margin:
                    fallback_price = wac / (1 - new_margin)
                    fallback_price = round(fallback_price * 4) / 4
                    if fallback_price > current_price:
                        return fallback_price, 'margin_pct_fallback'
        
        return current_price, 'unchanged (no tier above)'
    
    if action == 'decrease':
        new_price = find_next_price_below(current_price, row)
        if new_price < current_price:
            # Determine source
            source = 'tiers'
            
            # Apply commercial minimum floor
            commercial_min = row.get('commercial_min_price', 0)
            if pd.notna(commercial_min) and commercial_min > 0:
                new_price = max(new_price, commercial_min)
            
            return new_price, source
        return current_price, 'unchanged (no tier below)'
    
    return current_price, 'unchanged'

def get_initial_cart_rule(row, percentile_data, is_oos=False, is_zero_demand=False):
    """
    Calculate initial cart rule using percentile-based approach.
    
    Primary: Use percentiles from historical order data
    - Normal case: 95th percentile
    - OOS: 95th percentile
    - Zero Demand: 95th percentile
    - Low Stock (DOH <= 2): 50th percentile
    
    Fallback: If percentile data unavailable, use 10 (minimum for Module 2)
    """
    cohort_id = row.get('cohort_id')
    product_id = row.get('product_id')
    
    # Try to get percentile data
    if len(percentile_data) > 0:
        percentile_row = percentile_data[
            (percentile_data['cohort_id'] == cohort_id) & 
            (percentile_data['product_id'] == product_id)
        ]
        
        if len(percentile_row) > 0:
            # Special case: OOS - Use 95th percentile
            if is_oos:
                perc_95 = percentile_row.iloc[0]['perc_95']
                if pd.notna(perc_95) and perc_95 > 0:
                    return max(MIN_CART_RULE, min(MAX_CART_RULE, int(round(perc_95))))
            
            # Special case: Zero Demand - Use 95th percentile
            if is_zero_demand:
                perc_95 = percentile_row.iloc[0]['perc_95']
                if pd.notna(perc_95) and perc_95 > 0:
                    return max(MIN_CART_RULE, min(MAX_CART_RULE, int(round(perc_95))))
            
            # Special case: Low Stock (DOH <= 2) - Use 50th percentile
            doh = row.get('doh', 999)
            if doh <= LOW_STOCK_DOH_THRESHOLD:
                perc_50 = percentile_row.iloc[0]['perc_50']
                if pd.notna(perc_50) and perc_50 > 0:
                    return max(MIN_CART_RULE, min(MAX_CART_RULE, int(round(perc_50))))
            
            # Normal case: layer_1 for most SKUs, perc_95 for both-above-on-track
            combined = row.get('combined_status', 'No Data')
            yesterday = row.get('yesterday_status', 'No Data')
            if is_above_on_track(combined) and is_above_on_track(yesterday):
                target_col = 'perc_95'
            else:
                target_col = 'layer_1'
            
            value = percentile_row.iloc[0].get(target_col)
            if pd.notna(value) and value > 0:
                return max(MIN_CART_RULE, min(MAX_CART_RULE, int(round(value))))
    
    # Fallback: no percentile data available
    return MIN_CART_RULE

print("Status adjustment and price functions loaded.")


Status adjustment and price functions loaded.


In [5]:
# =============================================================================
# MAIN ENGINE: GENERATE INITIAL PRICE PUSH
# =============================================================================

def get_oos_price(row):
    """Get OOS price from effective_tiers based on ABC class.
    A -> P25, B -> P50, C -> P75. Lower percentile = more aggressive defense for high-velocity SKUs."""
    tiers = row.get('effective_tiers', [])
    if not isinstance(tiers, list) or len(tiers) == 0:
        return row.get('current_price', 0), 'unchanged_no_tiers'
    abc = str(row.get('abc_class', 'C')).strip().upper()
    if abc == 'A':
        p = np.percentile(tiers, 25)
        pct = 25
    elif abc == 'B':
        p = np.percentile(tiers, 50)
        pct = 50
    else:
        p = np.percentile(tiers, 75)
        pct = 75
    p = round(p * 4) / 4
    return p, f'oos_{abc.lower()}_class_p{pct}'

def find_price_n_steps_below(current_price, n_steps, row):
    """Find price N steps below current (iteratively find next tier below)."""
    price = current_price
    for _ in range(n_steps):
        next_price = find_next_price_below(price, row)
        if next_price >= price:  # No tier below found
            break
        price = next_price
    return price

def generate_initial_price_push(row, percentile_data):
    """
    Generate initial price push action for a single SKU.
    
    Logic:
    - Stocks = 0: Set to market_max or margin_max (highest price)
    - Zero demand + yesterday below on track: Go 2 steps below current
    - Zero demand + yesterday above on track: Keep current price
    - Otherwise: Adjust price relative to CURRENT price based on status
    """
    result = {
        'product_id': row.get('product_id'),
        'warehouse_id': row.get('warehouse_id'),
        'cohort_id': row.get('cohort_id'),
        'sku': row.get('sku'),
        'brand': row.get('brand'),
        'cat': row.get('cat'),
        'abc_class': row.get('abc_class', 'C'),
        'current_price': row.get('current_price'),
        'current_cart_rule': row.get('current_cart_rule'),
        'wac_p': row.get('wac_p'),
        'stocks': row.get('stocks', 0),
        'combined_status': row.get('combined_status'),
        'yesterday_status': row.get('yesterday_status'),
        'zero_demand': row.get('zero_demand', 0),
        'sensitivity': row.get('sensitivity', row.get('product_sensitivity')),
        'new_price': None,
        'new_cart_rule': None,
        'new_margin': None,
        'current_margin': None,
        'price_source': None,
        'price_action': None,
        'price_reason': None,
    }
    
    wac = row.get('wac_p', 0)
    current_price = row.get('current_price', 0)
    result['current_margin'] = calculate_margin(current_price, wac)
    yesterday_status = row.get('yesterday_status', 'No Data')
    
    # CASE 1: Out of Stock (stocks = 0) - Set to ABC-class percentile of effective_tiers
    # A -> P25, B -> P50, C -> P75
    if row.get('stocks', 0) == 0:
        oos_price, price_source = get_oos_price(row)
        abc = str(row.get('abc_class', 'C')).strip().upper()
        result['new_price'] = oos_price
        result['price_source'] = price_source
        result['price_action'] = f'oos_{abc.lower()}_class'
        result['price_reason'] = f'OOS - {abc} class set via {price_source}'

        result['new_cart_rule'] = get_initial_cart_rule(row, percentile_data, is_oos=True)  # OOS cart: 95th percentile
        result['new_margin'] = calculate_margin(result['new_price'], wac)
        return result
    
    # CASE 2: Zero Demand SKUs (has stock but no recent sales)
    if row.get('zero_demand', 0) == 1:
        yesterday_below = is_below_on_track(yesterday_status)
        yesterday_above = is_above_on_track(yesterday_status)
        
        if yesterday_below:
            # Yesterday below on track: Go 2 steps below current price
            new_price = find_price_n_steps_below(current_price, 2, row)
            
            # Apply commercial minimum floor
            commercial_min = row.get('commercial_min_price', 0)
            if pd.notna(commercial_min) and commercial_min > 0:
                new_price = max(new_price, commercial_min)
            
            result['new_price'] = new_price
            result['price_source'] = 'tiers'
            result['price_action'] = 'zero_demand_decrease'
            result['price_reason'] = f'Zero demand + yesterday below ({yesterday_status}) - 2 steps below'
        elif yesterday_above:
            # Yesterday above on track: Keep current price
            result['new_price'] = current_price
            result['price_source'] = 'unchanged'
            result['price_action'] = 'zero_demand_hold'
            result['price_reason'] = f'Zero demand + yesterday above ({yesterday_status}) - keep current'
        else:
            # Yesterday on track or no data: Go 1 step below
            new_price = find_next_price_below(current_price, row)
            commercial_min = row.get('commercial_min_price', 0)
            if pd.notna(commercial_min) and commercial_min > 0:
                new_price = max(new_price, commercial_min)
            
            result['new_price'] = new_price
            result['price_source'] = 'tiers'
            result['price_action'] = 'zero_demand_decrease'
            result['price_reason'] = f'Zero demand + yesterday on track ({yesterday_status}) - 1 step below'
        
        result['new_cart_rule'] = get_initial_cart_rule(row, percentile_data, is_zero_demand=True)  # Zero demand: 95th percentile
        result['new_margin'] = calculate_margin(result['new_price'], wac)
        return result
    
    # CASE 2.5: LOW STOCK PROTECTION (DOH <= 2 with demand)
    # Protect inventory until next receiving - no price reduction, cap cart at normal_refill
    # But still allow price INCREASE if status indicates strong performance
    doh = row.get('doh', 999)
    if doh <= LOW_STOCK_DOH_THRESHOLD and row.get('zero_demand', 0) == 0 and row.get('stocks', 0) > 0:
        combined_status = row.get('combined_status', 'No Data')
        normal_refill = row.get('normal_refill', 5) or 5
        
        # Check if we should increase price (status above on track)
        if is_above_on_track(combined_status) or combined_status == 'On Track':
            new_price = find_next_price_above(current_price, row)
            result['new_price'] = new_price
            result['price_source'] = 'tiers'
            result['price_action'] = 'low_stock_increase'
            result['price_reason'] = f'Low stock (DOH={doh:.1f}) + above on track ({combined_status}) - increase allowed'
        else:
            # Hold price - no reduction for low stock
            result['new_price'] = current_price
            result['price_source'] = 'unchanged'
            result['price_action'] = 'low_stock_hold'
            result['price_reason'] = f'Low stock (DOH={doh:.1f}) - hold price (no reduction allowed)'
        
        # Cap cart at 50th percentile for low stock
        result['new_cart_rule'] = get_initial_cart_rule(row, percentile_data)  # Will use 50th percentile for low stock
        result['new_margin'] = calculate_margin(result['new_price'], wac)
        return result
    
    # CASE 3: Normal SKUs - Determine price action based on status
    combined_status = row.get('combined_status', 'No Data')
    
    # Handle 'No Data' with stocks as Critical (below on track)
    if combined_status == 'No Data' and row.get('stocks', 0) > 0:
        combined_status = 'Critical'
    
    # Get price action (increase/decrease/hold)
    action, reason = get_price_action(combined_status, yesterday_status)
    
    # =========================================================================
    # MARKET SIGNAL OVERRIDES (Normal SKUs only)
    # Only apply when data quality is sufficient and volatility is low
    # =========================================================================
    trend_signal = row.get('trend_signal', 'NEUTRAL')
    signal_data_points = row.get('data_points_30d', 0)
    signal_volatility = row.get('volatility_pct', 0)
    market_signal_valid = (
        signal_data_points >= 10 
        and (pd.isna(signal_volatility) or signal_volatility <= 5)
        and trend_signal in ('UPTREND', 'STRONG UPTREND')
    )
    
    # Fallback: commercial price-up signal (tiered by magnitude)
    if not market_signal_valid:
        price_up_pct = row.get('commercial_price_up_pct', 0) or 0
        if price_up_pct >= 0.15:
            market_signal_valid = True
            trend_signal = 'STRONG UPTREND'
        elif price_up_pct >= 0.05:
            market_signal_valid = True
            trend_signal = 'UPTREND'
    
    # 2B: Yesterday above on track + market uptrend → increase instead of hold
    if market_signal_valid and action == 'hold':
        if is_above_on_track(yesterday_status):
            action = 'increase'
            reason = f'Yesterday above on track + market {trend_signal} - increase'
    
    # 2A: Increase + market uptrend → 2 steps up with above-market fallback
    if market_signal_valid and action == 'increase':
        first_step = find_next_price_above(current_price, row)
        if first_step > current_price:
            second_step = find_next_price_above(first_step, row)
            new_price = second_step if second_step > first_step else first_step
        else:
            new_price = current_price
        
        # Above-market fallback: if stuck at top of ladder, push beyond
        if new_price <= current_price:
            new_price = get_above_market_price(current_price, row)
        
        result['new_price'] = new_price
        tiers = row.get('effective_tiers', [])
        result['price_source'] = 'above_market' if (tiers and new_price > max(tiers)) else 'tiers'
        result['price_action'] = 'increase_market_boost'
        result['price_reason'] = f'{reason} + market {trend_signal} - boost'
        result['new_cart_rule'] = get_initial_cart_rule(row, percentile_data)
        result['new_margin'] = calculate_margin(new_price, wac)
        return result
    
    # Standard path (no market signal override)
    increase_pct = get_margin_increase_pct(combined_status, yesterday_status) if action == 'increase' else None
    new_price, price_source = apply_price_action(current_price, action, row, margin_increase_pct=increase_pct)
    result['new_price'] = new_price
    result['price_source'] = price_source
    result['price_action'] = action
    result['price_reason'] = reason
    result['new_cart_rule'] = get_initial_cart_rule(row, percentile_data)
    result['new_margin'] = calculate_margin(new_price, wac)
    
    return result

print("Main engine function loaded.")


Main engine function loaded.


In [6]:
# =============================================================================
# EXECUTE MODULE 2
# =============================================================================
print(f"Processing {len(df)} SKUs...")
print("="*60)

results = []
for idx, row in df.iterrows():
    result = generate_initial_price_push(row, df_percentiles)
    results.append(result)
    
    if (idx + 1) % 10000 == 0:
        print(f"Processed {idx + 1}/{len(df)} SKUs...")

df_results = pd.DataFrame(results)
print(f"\n✅ Processed {len(df_results)} SKUs")


Processing 29967 SKUs...


Processed 10000/29967 SKUs...


Processed 20000/29967 SKUs...



✅ Processed 29967 SKUs


In [7]:
# =============================================================================
# MARKET MAX CEILING (price <= max(effective_tiers) unless growing)
# Growing = combined_status AND yesterday_status both above on track
# commercial_min_price overrides this ceiling
# =============================================================================
print("Applying market max ceiling...")
ceiling_capped = 0
ceiling_current = 0
for idx in df_results.index:
    tiers = df_results.loc[idx, 'effective_tiers'] if 'effective_tiers' in df_results.columns else []
    if not isinstance(tiers, list) or len(tiers) == 0:
        continue
    market_max = max(tiers)
    combined = str(df_results.loc[idx, 'combined_status']).strip()
    yesterday = str(df_results.loc[idx, 'yesterday_status']).strip()
    is_growing = is_above_on_track(combined) and is_above_on_track(yesterday)
    if is_growing:
        continue
    new_price = df_results.loc[idx, 'new_price']
    current_price = df_results.loc[idx, 'current_price']
    price_to_check = new_price if pd.notna(new_price) else current_price
    if pd.notna(price_to_check) and price_to_check > market_max:
        reason = df_results.loc[idx, 'price_reason'] if pd.notna(df_results.loc[idx, 'price_reason']) else ''
        if pd.notna(new_price):
            df_results.at[idx, 'new_price'] = market_max
            df_results.at[idx, 'price_reason'] = f"{reason} | capped at market max ({new_price:.2f} -> {market_max:.2f})" if reason else f"capped at market max ({new_price:.2f} -> {market_max:.2f})"
            ceiling_capped += 1
        else:
            df_results.at[idx, 'new_price'] = market_max
            df_results.at[idx, 'price_action'] = 'market_max_cap'
            df_results.at[idx, 'price_reason'] = f"current price above market max ({current_price:.2f} -> {market_max:.2f})"
            ceiling_current += 1
print(f"  Market max ceiling: {ceiling_capped} new prices capped, {ceiling_current} current prices brought down")

# =============================================================================
# FIXED PRICE OVERRIDE (from Google Sheet)
# =============================================================================
df_fixed = get_fixed_prices()
df_results = df_results.merge(df_fixed, on='product_id', how='left')
has_fixed = df_results['fixed_price'].notna()
df_results.loc[has_fixed, 'new_price'] = df_results.loc[has_fixed, 'fixed_price']
df_results.loc[has_fixed, 'price_action'] = 'fixed_price'
df_results.loc[has_fixed, 'price_reason'] = 'Fixed price from Google Sheet'
df_results = df_results.drop(columns=['fixed_price'])
print(f"Fixed price override: {has_fixed.sum()} SKUs set to fixed price from Google Sheet")

# =============================================================================
# FIXED CART RULE OVERRIDE (from Google Sheet - Sheet2)
# =============================================================================
df_fixed_cart = get_fixed_cart_rules()
df_results = df_results.merge(df_fixed_cart, on='product_id', how='left')
has_fixed_cart = df_results['fixed_cart_rule'].notna()
df_results.loc[has_fixed_cart, 'new_cart_rule'] = df_results.loc[has_fixed_cart, 'fixed_cart_rule'].astype(int)
df_results = df_results.drop(columns=['fixed_cart_rule'])
print(f"Fixed cart rule override: {has_fixed_cart.sum()} SKUs set to fixed cart rule from Google Sheet")

Applying market max ceiling...
  Market max ceiling: 0 new prices capped, 0 current prices brought down
Fetching fixed prices from Google Sheet...


/home/ec2-user/service_account_key.json


  Loaded 12 fixed price SKUs
Fixed price override: 125 SKUs set to fixed price from Google Sheet
Fetching fixed cart rules from Google Sheet...


/home/ec2-user/service_account_key.json


  Loaded 0 fixed cart rule SKUs
Fixed cart rule override: 0 SKUs set to fixed cart rule from Google Sheet


In [8]:
# =============================================================================
# SUMMARY
# =============================================================================
print("="*60)
print("MODULE 2 SUMMARY")
print("="*60)

print(f"\nTotal SKUs processed: {len(df_results)}")
print(f"\nBy ABC Class:")
print(df_results['abc_class'].value_counts().to_string())
print(f"\nBy Price Source:")
print(df_results['price_source'].value_counts().to_string())

# Price change analysis
df_results['price_change'] = df_results['new_price'] - df_results['current_price']
df_results['price_change_pct'] = (df_results['price_change'] / df_results['current_price'] * 100).round(2)

print(f"\nPrice Change Distribution:")
print(f"  Increases: {len(df_results[df_results['price_change'] > 0])}")
print(f"  Decreases: {len(df_results[df_results['price_change'] < 0])}")
print(f"  No change: {len(df_results[df_results['price_change'] == 0])}")
print(f"\nAvg price change: {df_results['price_change_pct'].mean():.2f}%")


MODULE 2 SUMMARY

Total SKUs processed: 29967

By ABC Class:
abc_class
C    24405
B     4814
A      748

By Price Source:
price_source
oos_c_class_p75              9656
tiers                        6598
unchanged (no tier below)    5977
unchanged                    3746
unchanged_no_tiers           2421
oos_b_class_p50              1169
margin_pct_fallback           268
oos_a_class_p25                80
above_market                   39
unchanged (no tier above)      13

Price Change Distribution:
  Increases: 7597
  Decreases: 7009
  No change: 15361

Avg price change: 2.10%


In [9]:
# =============================================================================
# EXPORT RESULTS
# =============================================================================
output_cols = [
    'product_id', 'warehouse_id', 'cohort_id', 'sku', 'brand', 'cat', 'abc_class', 'sensitivity',
    'stocks', 'zero_demand', 'combined_status', 'yesterday_status',
    'current_price', 'new_price', 'price_change', 'price_change_pct',
    'wac_p', 'current_margin', 'new_margin','current_cart_rule',
    'new_cart_rule', 'price_action', 'price_source', 'price_reason'
]

# Filter to only columns that exist
output_cols = [c for c in output_cols if c in df_results.columns]

# Drop duplicates before saving
df_output = df_results[output_cols].drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
# Save df_output state before any manipulation for Slack upload later
temp_df_for_slack = df_output.copy()
print(f"\n✅ Saved {len(temp_df_for_slack)} rows for Slack upload")
print(f"Total records: {len(df_output)} (after removing {len(df_results) - len(df_output)} duplicates)")



✅ Saved 29967 rows for Slack upload
Total records: 29967 (after removing 0 duplicates)


In [10]:
# =============================================================================
# PUSH CART RULES & PRICES
# =============================================================================
# Push cart rules FIRST, then prices
# If cart rules fail for certain cohorts, skip those cohorts for prices

%run push_cart_rules_handler.ipynb
%run push_prices_handler.ipynb
pus = get_packing_units()

# ⚠️ MODE CONFIGURATION:
# - 'testing' (default): Prepare files but DON'T upload to API
# - 'live': Prepare files AND upload to MaxAB API
PUSH_MODE = 'live'  # Change to 'live' when ready to push

# =============================================================================
# STEP 1: Push Cart Rules First
# =============================================================================
print("\n" + "="*70)
print("STEP 1: PUSHING CART RULES")
print("="*70)

cart_result = push_cart_rules(df_output, pus, source_module='module_2', mode=PUSH_MODE)

print(f"\n{'='*60}")
print("CART RULES RESULT")
print(f"{'='*60}")
print(f"Mode: {cart_result['mode']}")
print(f"Cart rule changes: {cart_result['cart_rule_changes']}")
print(f"Pushed: {cart_result['pushed']}")
print(f"Failed: {cart_result['failed']}")
if cart_result['failed_cohorts']:
    print(f"⚠️ Failed cohorts: {cart_result['failed_cohorts']}")

# =============================================================================
# STEP 2: Push Prices (skip failed cohorts)
# =============================================================================
print("\n" + "="*70)
print("STEP 2: PUSHING PRICES")
print("="*70)

# Get failed cohorts from cart rules to skip in price push
failed_cohorts = cart_result.get('failed_cohorts', [])

# Call push_prices with the results, skipping failed cohorts
push_result = push_prices(df_output, pus, source_module='module_2', mode=PUSH_MODE, skip_cohorts=failed_cohorts)

print(f"\n{'='*60}")
print("PRICES RESULT")
print(f"{'='*60}")
print(f"Mode: {push_result['mode']}")
print(f"Source: {push_result['source_module']}")
print(f"Timestamp: {push_result['timestamp']}")
print(f"Total received: {push_result['total_received']}")
print(f"Price changes: {push_result['price_changes']}")
print(f"Pushed: {push_result['pushed']}")
print(f"Skipped: {push_result['skipped']}")
print(f"Failed: {push_result['failed']}")
if push_result.get('skipped_cohorts'):
    print(f"⚠️ Skipped cohorts (cart rules failed): {push_result['skipped_cohorts']}")

# =============================================================================
# STEP 3: MIRROR TO NON-FOOD COHORTS (isolated - failures won't stop main module)
# =============================================================================
# print(f"\n{'='*70}")
# print("STEP 3: MIRROR TO NON-FOOD COHORTS")
# print(f"{'='*70}")
# try:
#     %run non_food_cohorts_push.ipynb
#     nf_result = push_to_non_food_cohorts(df_output, source_module='module_2', mode=PUSH_MODE)
#     send_summary_to_slack(nf_result)
# except Exception as _nf_e:
#     import traceback as _tb
#     _err = _tb.format_exc()
#     print(f"Non-food cohorts push FAILED (main module continues): {_nf_e}")
#     try:
#         from common_functions import send_text_slack as _slack
#         _slack('new-pricing-logic',
#                f"*Non-Food Cohorts Push FAILED*\n*Source:* `module_2`\n*Error:* `{_nf_e}`\n```\n{_err[-1000:]}\n```")
#     except Exception:
#         pass


Push Cart Rules Handler loaded at 2026-04-21 08:19:07 Cairo time


✓ API credentials loaded successfully
Push Prices Handler loaded at 2026-04-21 08:19:08 Cairo time


✓ API credentials loaded successfully


✓ Google Sheets client initialized
Fetching packing_units ...


  Loaded 36311 records

STEP 1: PUSHING CART RULES

🚀 MODE: LIVE
   Files will be prepared AND uploaded to API

PUSH CART RULES - Source: module_2
Total received: 29967
Cart rule changes to push: 17750
Skipped (no change): 12217

Cart rule changes summary:
  Increases: 2150
  Decreases: 15600

📋 Prepared 19644 packing unit cart rules


...

  Split into 1 chunks (size: 4000)


  Saving chunks:   0%|          | 0/1 [00:00<?, ?it/s]

  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 14.08it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 29366
Total failed: 0
  Cleanup: removed 31 temporary .xlsx files from 2 folder(s)

PRICES RESULT
Mode: live
Source: module_2
Timestamp: 2026-04-21 08:19:54
Total received: 29967
Price changes: 14606
Pushed: 29366
Skipped: 15361
Failed: 0


In [11]:
# =============================================================================
# UPLOAD RESULTS TO SNOWFLAKE AND SEND SLACK NOTIFICATION
# =============================================================================
from common_functions import upload_dataframe_to_snowflake, send_text_slack, send_file_slack

# Add created_at as DATE (module runs once per day at 8 AM)
df_output['created_at'] = datetime.now(CAIRO_TZ).date()

# Upload to Snowflake
print("\n" + "="*60)
print("UPLOADING RESULTS TO SNOWFLAKE")
print("="*60)

upload_status = upload_dataframe_to_snowflake(
    "Egypt", 
    df_output, 
    "MATERIALIZED_VIEWS", 
    "pricing_initial_push", 
    "append", 
    auto_create_table=True, 
    conn=None
)

# Prepare Slack notification
prices_pushed = push_result.get('pushed', 0)
prices_failed = push_result.get('failed', 0)
cart_rules_pushed = cart_result.get('pushed', 0)
cart_rules_failed = cart_result.get('failed', 0)

if upload_status:
    slack_message = f"""✅ *Module 2 - Initial Price Push Completed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Completed at: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
🔧 Mode: {PUSH_MODE.upper()}

📊 *Results:*
• Total SKUs processed: {len(df_output):,}
• Price changes: {push_result.get('price_changes', 0):,}
• Cart rule changes: {cart_result.get('cart_rule_changes', 0):,}

📤 *Push Status:*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed

🗄️ Results uploaded to: MATERIALIZED_VIEWS.pricing_initial_push"""
    
    send_text_slack('new-pricing-logic', slack_message)
    print("✅ Slack notification sent!")
    
    # Send output file to Slack after the text message (using saved copy before manipulation)
    SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'
    send_file_slack(
        temp_df_for_slack, 
        f'📎 Module 2 Output: {len(temp_df_for_slack)} SKUs processed', 
        SLACK_CHANNEL_ID,
        filename=f'module2_initial_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Output file sent to Slack")
    
    print(f"✅ {len(df_output)} records uploaded to Snowflake")
else:
    error_message = f"""❌ *Module 2 - Initial Price Push Failed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Failed at: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
⚠️ Upload to Snowflake failed - please check logs

📤 *Push Status (before upload failure):*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed"""
    
    send_text_slack('new-pricing-logic', error_message)
    print("❌ Error notification sent to Slack!")
    
    # Still send output file even on error for debugging (using saved copy before manipulation)
    send_file_slack(
        temp_df_for_slack, 
        f'⚠️ Module 2 ERROR: {len(temp_df_for_slack)} SKUs', 
        SLACK_CHANNEL_ID,
        filename=f'module2_initial_ERROR_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Error file sent to Slack")



UPLOADING RESULTS TO SNOWFLAKE


/home/ec2-user/service_account_key.json


/home/ec2-user/service_account_key.json


Message Sent
✅ Slack notification sent!


/home/ec2-user/service_account_key.json


File module2_initial_20260421_0828.xlsx sent to Slack
✅ Output file sent to Slack
✅ 29967 records uploaded to Snowflake
